In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("file_date", "2024-12-16", "File Date")
file_date = dbutils.widgets.get("file_date")

In [0]:
from pyspark.sql.functions import col, current_timestamp, lit
from delta.tables import DeltaTable

In [0]:
production_company_df = (
    spark.table("databricks_ws.bronze.production_companies")
        .filter(col("file_date") == file_date)
    )

In [0]:
production_company_selected_df = (
    production_company_df
    .select(
        col("_c0").cast("int").alias("company_id"),
        col("_c1").cast("string").alias("company_name"),
        col("file_date")
    )
)

In [0]:
production_companies_final_df = (
    add_ingestion_date(production_company_selected_df)
    .withColumn("environment", lit("Production"))
)

In [0]:
table_name = "databricks_ws.silver.production_companies"
if spark.catalog.tableExists(table_name):

    deltaTable = DeltaTable.forName(spark, table_name)

    deltaTable.alias("tgt") \
        .merge(
            production_companies_final_df.alias("src"),
            "tgt.company_id = src.company_id"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
else:
    production_companies_final_df.write \
        .format("delta") \
        .saveAsTable(table_name)